# All Methods + Full Data + Backtest — BTC 4-Hourly

The complete comparison. **13 variants × 2 windows × 2 target modes (3 seeds each) = 8 experiments**, then an automatic **portfolio backtest of every return-mode variant** with cross-variant comparison figures.

## Experiments

| # | Family | Window | Mode | Base config |
|---|---|---|---|---|
| 1 | OHLC (7 var) | 365d | price | `btc_ohlc.yaml` |
| 2 | OHLC (7 var) | 365d | return | `btc_ohlc_return.yaml` |
| 3 | Hier (6 var) | 365d | price | `btc_hier.yaml` |
| 4 | Hier (6 var) | 365d | return | `btc_hier_return.yaml` |
| 5 | OHLC (7 var) | **full (~13k bars)** | price | `btc_ohlc_full.yaml` |
| 6 | OHLC (7 var) | **full** | return | `btc_ohlc_return_full.yaml` |
| 7 | Hier (6 var) | **full** | price | `btc_hier_full.yaml` |
| 8 | Hier (6 var) | **full** | return | `btc_hier_return_full.yaml` |

The **full-data** experiments (5–8) drop the 365-day filter → ~6× more training data, to test whether the persistence collapse is caused by data starvation.

## What you get
- Per-experiment comparison figures (metric bars, box plots, radar) via `visualize_results.py`
- One **aggregate table** (mean ± std per variant per experiment) + a **signal-quality table** (corr, dir-agree)
- For every **return-mode** experiment: a portfolio backtest of all variants with **backtest-comparison bars**, **signal-quality scatter grid**, and **equity overlay vs buy & hold**

## Runtime
**Very heavy.** 8 experiments, the full-data ones are ~6× longer per epoch. Expect many hours on a T4. Run unattended; keep the tab alive. You can comment out experiments 5–8 for a first pass.

**Requires:** `lunarcrush_btc_4hour_full.csv` in Google Drive.

## 1. Setup

In [ ]:
import os
if not os.path.exists('/content/thesis'):
    !git clone https://github.com/BerkayClik/thesis.git /content/thesis
%cd /content/thesis
!git pull

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

In [ ]:
# Copy the BTC 4-hour cache CSV from Drive (recursive search).
import os, shutil, glob
os.makedirs('data/cache', exist_ok=True)
DRIVE_DATA_DIR = '/content/drive/MyDrive/thesis_data'   # <-- the folder you uploaded to
copied = 0
if os.path.isdir(DRIVE_DATA_DIR):
    for src in glob.glob(os.path.join(DRIVE_DATA_DIR, '**/lunarcrush_btc_4hour*.csv'), recursive=True):
        shutil.copy(src, os.path.join('data/cache', os.path.basename(src))); copied += 1
        print('copied', os.path.basename(src))
if copied == 0:
    print(f'No BTC 4-hour CSV found under {DRIVE_DATA_DIR} (searched recursively).')
!ls -la data/cache/lunarcrush_btc_4hour*.csv 2>/dev/null || echo 'missing BTC 4h cache'

In [ ]:
!pip install -q yfinance scipy seaborn uv
import torch
print('PyTorch:', torch.__version__, '| CUDA:', torch.cuda.is_available(),
      '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
os.environ['PYTHONPATH'] = '/content/thesis'
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'

In [ ]:
# Build the isolated vectorbt backtest env once (needs numpy<2)
!uv venv --python 3.11 .venv-backtest
!uv pip install --python .venv-backtest -r requirements-backtest.txt
!.venv-backtest/bin/python scripts/backtest_env_smoke.py

## 2. Run the experiments

Each cell is one experiment (13 variants split as 7 OHLC + 6 hierarchical, × 3 seeds). Experiments 5–8 use the full ~13k-bar dataset and take much longer — comment them out for a quick first pass.

In [ ]:
EXP = [
    # (base_config, experiment_config) ; experiment family decides variant set
    ('btc_ohlc',             '4hourly_comparison_3seed'),    # 1 OHLC price 365d
    ('btc_ohlc_return',      '4hourly_comparison_3seed'),    # 2 OHLC return 365d
    ('btc_hier',             '4hourly_hierarchical_3seed'),  # 3 Hier price 365d
    ('btc_hier_return',      '4hourly_hierarchical_3seed'),  # 4 Hier return 365d
    ('btc_ohlc_full',        '4hourly_comparison_3seed'),    # 5 OHLC price full
    ('btc_ohlc_return_full', '4hourly_comparison_3seed'),    # 6 OHLC return full
    ('btc_hier_full',        '4hourly_hierarchical_3seed'),  # 7 Hier price full
    ('btc_hier_return_full', '4hourly_hierarchical_3seed'),  # 8 Hier return full
]
for i, (base, exp) in enumerate(EXP, 1):
    print(f'{i}. base=configs/data/4hourly/{base}.yaml  exp=configs/experiments/{exp}.yaml')

> **✅ Resumable / disconnect-proof.** The run cell below:
>
> - **Saves each experiment to Google Drive the moment it finishes** (a fixed `.../checkpoint/` folder), so a Colab disconnect never loses completed work.
> - **Skips experiments that are already done** on restart.
>
> **If Colab disconnects mid-run:** just reconnect, re-run cells 1–6 (setup), then re-run this run cell again — it restores finished experiments from Drive and continues from where it stopped. Repeat until every experiment prints either fresh results or `[skip] already finished`.
>
> **Tip:** keep the browser tab active (free Colab disconnects idle sessions after ~90 min). Running overnight may take several reconnect cycles — that's expected and safe now.

In [ ]:
# Resumable run loop (survives Colab disconnects).
#  - Restores any prior results from a FIXED Drive checkpoint folder.
#  - Skips experiments already finished (results_dir has a non-intermediate JSON).
#  - Saves each experiment's results to Drive immediately after it completes.
# Re-run this cell after a disconnect; it continues where it left off.
import os, glob, shutil

CKPT_DRIVE = '/content/drive/MyDrive/thesis_results_all_full_backtest_4h/checkpoint'   # FIXED folder (not timestamped) so resume works
os.makedirs(CKPT_DRIVE, exist_ok=True)

def _result_dir(base):
    import yaml
    c = yaml.safe_load(open(f'configs/data/4hourly/' + base + '.yaml'))
    return c['output']['results_dir']

def _is_done(rdir):
    return any('intermediate' not in f for f in glob.glob(f'{rdir}/*.json'))

# 1) Restore prior results from Drive checkpoint into the local results tree
for base, _ in EXP:
    rdir = _result_dir(base)
    saved = os.path.join(CKPT_DRIVE, os.path.basename(rdir))
    if os.path.isdir(saved) and not os.path.exists(rdir):
        shutil.copytree(saved, rdir, dirs_exist_ok=True)
        print('restored from Drive:', os.path.basename(rdir))

# 2) Run each experiment, skipping ones already done; save to Drive after each
for i, (base, exp) in enumerate(EXP, 1):
    rdir = _result_dir(base)
    print(f'\n{"="*70}\nEXPERIMENT {i}/8: {base}  /  {exp}\n{"="*70}')
    if _is_done(rdir):
        print('  [skip] already finished ->', rdir)
        continue
    !python experiments/run_experiments.py --base-config configs/data/4hourly/{base}.yaml --experiment-config configs/experiments/{exp}.yaml
    # immediate Drive save (so a later disconnect cannot lose this experiment)
    if _is_done(rdir):
        shutil.copytree(rdir, os.path.join(CKPT_DRIVE, os.path.basename(rdir)), dirs_exist_ok=True)
        print('  saved to Drive checkpoint:', os.path.basename(rdir))


## 3. Aggregate tables (all 8 experiments)

In [ ]:
import json, glob, os
import numpy as np, pandas as pd

RESULT_DIRS = {
    'OHLC·price·365':  'experiments/results/4hourly_btc_ohlc',
    'OHLC·return·365': 'experiments/results/4hourly_btc_ohlc_return',
    'Hier·price·365':  'experiments/results/4hourly_btc_hier',
    'Hier·return·365': 'experiments/results/4hourly_btc_hier_return',
    'OHLC·price·full':  'experiments/results/4hourly_btc_ohlc_full',
    'OHLC·return·full': 'experiments/results/4hourly_btc_ohlc_return_full',
    'Hier·price·full':  'experiments/results/4hourly_btc_hier_full',
    'Hier·return·full': 'experiments/results/4hourly_btc_hier_return_full',
}
METRICS = ['mape', 'directional_accuracy', 'sharpe_ratio',
           'directional_accuracy_3class', 'sharpe_ratio_3class']

def latest_json(d):
    js = [f for f in glob.glob(f'{d}/*.json') if 'intermediate' not in f]
    return max(js, key=os.path.getmtime) if js else None

agg_rows, sig_rows = [], []
for exp_name, d in RESULT_DIRS.items():
    jf = latest_json(d)
    if not jf:
        print(f'[skip] {exp_name}: no results')
        continue
    res = json.load(open(jf))
    for variant, vdata in res['model_results'].items():
        agg = vdata.get('aggregated', {})
        row = {'experiment': exp_name, 'variant': variant}
        for m in METRICS:
            row[m] = agg.get(m, {}).get('mean', np.nan)
        agg_rows.append(row)
        tm = vdata['individual_runs'][0]['test_metrics']
        if tm.get('predictions'):
            pr = np.array(tm['predictions'], float) / np.array(tm['prev_closes'], float) - 1
            tr = np.array(tm['targets'], float) / np.array(tm['prev_closes'], float) - 1
            corr = np.corrcoef(pr, tr)[0, 1] if pr.std() and tr.std() else np.nan
            da = (np.sign(pr) == np.sign(tr)).mean() * 100
            sig_rows.append({'experiment': exp_name, 'variant': variant,
                             'corr': round(corr, 4), 'dir_agree_%': round(da, 1)})

agg_df = pd.DataFrame(agg_rows).round(3)
sig_df = pd.DataFrame(sig_rows)
pd.set_option('display.width', 220, 'display.max_rows', 200)
print('=== Metrics (mean over 3 seeds) ==='); display(agg_df)
print('=== Signal quality (corr, dir-agree) ==='); display(sig_df)

## 4. Per-experiment comparison figures

Standard multi-variant figures (metric bars, box plots, radar) for each experiment.

In [ ]:
from IPython.display import Image, display
for exp_name, d in RESULT_DIRS.items():
    jf = latest_json(d)
    if not jf:
        continue
    out = f'{d}/figures'
    print(f'\n=== {exp_name} ===')
    !python experiments/visualize_results.py --results "{jf}" --output "{out}" 2>&1 | tail -1
    for fig in ['metric_comparison.png', 'box_plots.png', 'radar_chart.png']:
        p = f'{out}/{fig}'
        if os.path.exists(p):
            display(Image(filename=p, width=820))

## 5. Backtest every return-mode variant

For each of the 4 return-mode experiments, backtest all variants (seed 42, next-bar-open execution, fees+slippage) and render: backtest-comparison bars, signal-quality scatter grid, and equity overlay vs buy & hold. Runs in the isolated vectorbt env.

In [ ]:
RETURN_DIRS = [
    'experiments/results/4hourly_btc_ohlc_return',
    'experiments/results/4hourly_btc_hier_return',
    'experiments/results/4hourly_btc_ohlc_return_full',
    'experiments/results/4hourly_btc_hier_return_full',
]
for d in RETURN_DIRS:
    if not glob.glob(f'{d}/*_seed42_predictions.csv'):
        print(f'[skip] {d}: no seed-42 predictions')
        continue
    print(f'\n{"="*70}\nBACKTEST ALL: {os.path.basename(d)}\n{"="*70}')
    !.venv-backtest/bin/python scripts/backtest_all.py \
        --results-dir "{d}" \
        --ohlc data/cache/lunarcrush_btc_4hour_full.csv \
        --seed 42 --fees 0.001 --slippage 0.0005

In [ ]:
# Display the backtest comparison figures for every return-mode experiment
for d in RETURN_DIRS:
    bt = f'{d}/backtest_all'
    if not os.path.isdir(bt):
        continue
    print(f'\n=== {os.path.basename(d)} ===')
    for png in sorted(glob.glob(f'{bt}/*.png')):
        print(png)
        display(Image(filename=png, width=900))

## 6. Save everything to Google Drive

In [ ]:
import shutil
from datetime import datetime
GDRIVE = '/content/drive/MyDrive/thesis_results_all_full_backtest_4h'
run_dir = f"{GDRIVE}/{datetime.now().strftime('%Y%m%d_%H%M%S')}"
os.makedirs(run_dir, exist_ok=True)
for exp_name, d in RESULT_DIRS.items():
    if os.path.exists(d):
        shutil.copytree(d, f'{run_dir}/{os.path.basename(d)}', dirs_exist_ok=True)
agg_df.to_csv(f'{run_dir}/metrics_table.csv', index=False)
sig_df.to_csv(f'{run_dir}/signal_quality_table.csv', index=False)
print(f'Saved all results + tables to: {run_dir}')

## Notes

- **Full-data vs 365d** is the key experiment for the 'is it data-starved?' question — compare corr / dir-agree of experiments 6 & 8 (full return) against 2 & 4 (365d return) in the signal-quality table.
- The legacy `test_metrics.sharpe_ratio` is the toy sign-based Sharpe. The real fee-aware Sharpe is in each `backtest_all/*_summary.csv`.
- A non-trading variant (no entries) shows `sharpe=n/a` — expected, not an error.
- To try the directional loss, add `loss_type: directional_mse` + `lambda_dir: 0.1` to a return-mode base config's `training:` block (a local sweep showed corr degrades as lambda rises, so keep it small).